In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import scanpy as sc
import pandas as pd
import anndata as ad
import numpy as np
import cell2location

In [2]:
in_dir = "../../results/02_10x_visium/01_filter_data"
out_dir = '../../results/03_kidney_atlas/06_run_cell2location'

os.makedirs(out_dir, exist_ok=True)

In [3]:
sample_list = ['15837', '15852', '16512', '21022', '8262', 
               'B23-03794-5', 'B23-05785-7-1', 'B23-05785-7-2', 
               'X1842', 'X1845', 'X1899', 'X1912']

In [ ]:
for sample in sample_list:
    adata_vis = sc.read_h5ad(f'{in_dir}/{sample}.h5ad')
    
    adata_vis.var_names_make_unique()

    # find mitochondria-encoded (MT) genes
    adata_vis.var['MT_gene'] = [gene.startswith('MT-') for gene in adata_vis.var_names]

    # remove MT genes for spatial mapping (keeping their counts in the object)
    adata_vis.obsm['MT'] = adata_vis[:, adata_vis.var['MT_gene'].values].X.toarray()
    adata_vis = adata_vis[:, ~adata_vis.var['MT_gene'].values]
    
    for subclass in ["l1", "l2"]:
        df_avg_exp = pd.read_csv(f'../../results/03_kidney_atlas/05_find_markers/marker_genes_subclass_{subclass}.csv', index_col=0)
        df_avg_exp.columns = [col.replace('/', '_') for col in df_avg_exp.columns]
        df_avg_exp.columns = [col.replace('-', '_') for col in df_avg_exp.columns]
        
        # find shared genes and subset both anndata and reference signatures
        intersect = np.intersect1d(adata_vis.var_names, df_avg_exp.index)
        adata_vis = adata_vis[:, intersect].copy()
        inf_aver_subset = df_avg_exp.loc[intersect, :].copy()

        # prepare anndata for cell2location model
        cell2location.models.Cell2location.setup_anndata(adata=adata_vis)
        
        # create and train the model
        mod = cell2location.models.Cell2location(
            adata_vis, 
            cell_state_df=inf_aver_subset,
            N_cells_per_location=30, # number of cells per spot
            detection_alpha=20
        )
        
        mod.train(max_epochs=30000, batch_size=None, train_size=1)
        
        # In this section, we export the estimated cell abundance (summary of the posterior distribution).
        adata_vis = mod.export_posterior(
            adata_vis, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs}
        )
        
        adata_vis.obs[adata_vis.uns['mod']['factor_names']] = adata_vis.obsm['q05_cell_abundance_w_sf']

        adata_vis.write(f'{out_dir}/{sample}_subclass_{subclass}.h5ad')

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.ht

Epoch 30000/30000: 100%|██████████| 30000/30000 [29:05<00:00, 16.92it/s, v_num=1, elbo_train=2.44e+5]

`Trainer.fit` stopped: `max_epochs=30000` reached.


Epoch 30000/30000: 100%|██████████| 30000/30000 [29:05<00:00, 17.19it/s, v_num=1, elbo_train=2.44e+5]

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


...

Epoch 30000/30000: 100%|██████████| 30000/30000 [31:05<00:00, 16.08it/s, v_num=1, elbo_train=2.51e+5]

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.



Sampling global variables, sample: 100%|██████████| 999/999 [00:12<00:00, 77.40it/s]


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 294/30000:   1%|          | 293/30000 [00:17<32:48, 15.09it/s, v_num=1, elbo_train=6.04e+5]